# 04 · Embeddings from the pretrained model

**NVIDIA TFM Blueprint, notebook 4 of 5.** After pretraining, the model's
state at a row's last token summarizes that row: its embedding. This notebook
embeds each scored transaction two ways, with the customer's recent history
(the kit's) and on its own (v1's), and looks at both.

Every notebook runs on CPU in about a minute on a small generated book, so
you can read the whole series without downloading anything. To see real rows,
build the TabFormer splits once (notebook 01 shows how) and set
`TFM_TABFORMER_ROOT`; the notebooks then load 24 TabFormer users with
`config.small.yaml`. The full-size runs are recipe commands, listed in notebooks 03
and 05, not notebook cells.

In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

import os

import matplotlib.pyplot as plt
import pandas as pd

from tfm.data.collate import minibatches
from tfm.evaluation.representation import project_2d
from tfm.inference.embedding import extract_embeddings
from tfm.runtime import build_model, resolve_device
from tfm.training.trainer import train_model

from tfm.config import load_recipe_config
from tfm.data.adapters.paths import ADAPTER_ENV
from tfm.extras.capability import extra_available
from tfm.runtime import prepare_data
from tfm.runtime.common import recipe_dir
from tfm.training.reproducibility import seed_everything

USING_TABFORMER = bool(os.environ.get(ADAPTER_ENV["tabformer"]))
CONFIG = recipe_dir("blueprint_v1") / ("config.small.yaml" if USING_TABFORMER else "config.smoke.yaml")
config = load_recipe_config(CONFIG)
if config.model.backbone == "huggingface" and not extra_available("transformers"):
    print("transformers extra not installed; using the eager decoder (uv sync --extra huggingface)")
    composition = config.model.composition.model_copy(update={"position": "nope"})
    model_cfg = config.model.model_copy(update={"backbone": "eager", "composition": composition})
    config = config.model_copy(update={"model": model_cfg})
seed_everything(config.seed)
prepared = prepare_data(config)
tokenizer = prepared.tokenizer
train_batch, val_batch, test_batch = prepared.batches
print("config", CONFIG.name, "| adapter", config.data.adapter, "| backbone", config.model.backbone)
from tfm.runtime.evaluate import event_entity_split, history_window_tasks

split = event_entity_split(prepared, config.seed)
tasks = history_window_tasks(prepared, config, split)
rows, control = tasks["history_window"], tasks["single_event"]

## 1. Train, then embed each transaction with its history

`extract_embeddings` runs the model over each row of notebook 01 and keeps the
state at its last real token (`LastTokenPool`). The row ends at the transaction
being scored, so the embedding is that transaction read in the context of the
customer's previous ones. Each embedding keeps its customer and row ids, so
notebook 05's feature sets stay aligned by construction.

In [ ]:
device = resolve_device(config.runtime.device)
model = build_model(config, tokenizer.vocab_size)
train_model(
    model,
    minibatches(train_batch, config.training.batch_size),
    max_steps=config.training.max_steps,
    learning_rate=config.training.learning_rate,
    seed=config.seed,
    device=device,
)
history_emb = extract_embeddings(model, rows.train_batch, device=device)
print("one embedding per scored transaction:", history_emb.vectors.shape)
pd.DataFrame({"customer": history_emb.entity_ids, "row": history_emb.sequence_ids, "fraud": rows.y_train}).head()

## 2. v1's embedding: the transaction on its own

v1 embedded each transaction alone: `<bos>`, its tokens, `<eos>`, and the state
at the end. The model sees nothing before that transaction, so the embedding
can't use what pretraining learned about sequences. The kit keeps it as a
control on the same rows: comparing the two measures what the history adds.

In [ ]:
single_emb = extract_embeddings(model, control.train_batch, device=device)
print("history rows", tuple(rows.train_batch.input_ids.shape), "| single-transaction rows", tuple(control.train_batch.input_ids.shape))
print("real tokens per row: history", rows.train_batch.attention_mask.sum(1).float().mean().item(),
      "| single", control.train_batch.attention_mask.sum(1).float().mean().item())

## 3. A look at the embeddings

A 2-D projection is for exploration only. It can't show whether the embeddings
help a fraud model; notebook 05 measures that. PCA is the default because it is
linear and deterministic. UMAP, v1's choice, is used instead when `umap-learn`
is installed; distances between its clusters mean little.

In [ ]:
method = "umap" if extra_available("umap") and len(rows.y_train) >= 10 else "pca"
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, table) in zip(axes, (("with history", history_emb), ("transaction alone", single_emb)), strict=True):
    xy = project_2d(table.vectors, method=method, seed=config.seed)
    for value, label in ((0, "no fraud"), (1, "fraud")):
        keep = rows.y_train == value
        ax.scatter(xy[keep, 0], xy[keep, 1], s=16, alpha=0.6, label=label)
    ax.set_title(f"{name}, {method.upper()}")
    ax.legend()
fig.tight_layout()
plt.show()

## 4. Measuring what the plot suggests

A picture of clusters is easy to over-read: two or three directions show a
fraction of the space, overlapping points hide each other, and the axis scale
changes how tight a cluster looks. So measure the same question in the full
space and in a 3-D projection (`project(..., dims=3)`):

- **silhouette by customer**: how much closer each row is to its own
  customer's rows than to the nearest other customer's, from −1 to 1. Above 0,
  rows sit with their customer; near 0, customers overlap.
- **within ÷ between**: a row's average distance from its customer's centre,
  over the average distance between customer centres. Below 1, customers are
  tighter than they are far apart.
- **nearest row, same customer**: how often a row's nearest embedding belongs
  to the same customer.

With history, rows of one customer share most of their tokens, so they group
even under random weights; grouping by customer is not, by itself, evidence of
learning, and it is not fraud detection. Notebook 05 asks whether the
embedding adds anything to the raw fields.

In [ ]:
import numpy as np
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

from tfm.evaluation.representation import project

measures = {}
for name, table in (("with history", history_emb), ("transaction alone", single_emb)):
    vectors, customers = table.vectors, np.asarray(table.entity_ids)
    names = np.unique(customers)
    if not 2 <= len(names) < len(vectors):
        print(f"{name}: too few customers or rows to measure")
        continue
    centres = np.stack([vectors[customers == c].mean(axis=0) for c in names])
    within = np.linalg.norm(vectors - centres[np.searchsorted(names, customers)], axis=1).mean()
    between = np.mean([np.linalg.norm(a - b) for i, a in enumerate(centres) for b in centres[i + 1:]])
    _, nearest = NearestNeighbors(n_neighbors=2).fit(vectors).kneighbors(vectors)
    measures[name] = {
        f"silhouette by customer, {vectors.shape[1]}-D": silhouette_score(vectors, customers),
        "silhouette by customer, 3-D PCA": silhouette_score(
            project(vectors, dims=3, seed=config.seed), customers
        ),
        "within ÷ between": within / between,
        "nearest row, same customer": (customers[nearest[:, 1]] == customers).mean(),
    }
pd.DataFrame(measures).T.round(2)

> **What changed from v1**
>
> - **With history.** v1 pretrained on long windows but embedded one
>   transaction with no history. The kit embeds the transaction at the end of
>   the customer's recent history, and keeps v1's way as a control.
> - **Ids travel with vectors.** v1 joined embeddings back to rows through a
>   separate `row_ids.npy`. `EmbeddingTable` keeps ids, labels, and raw features
>   together.
> - **No cuML required.** PCA runs on CPU; UMAP is an optional extra.
> - **Measured, not only plotted.** The silhouette and distance ratios above
>   test what the projection suggests, in the full space.

## Takeaways

- An embedding is the model's state at the last real token.
- Ending each row at the scored transaction gives one embedding per
  transaction that still sees the history.
- Judge embeddings by a measurement and a probe, not a scatter plot.

Code: `tfm.inference.embedding`, `tfm.evaluation.representation`. Recipe:
`uv run tfm infer --recipe blueprint_v1`.

Next: [05 · Fraud detection](05_xgboost_fraud_detection.ipynb).